In [1]:
import os, re, glob, cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import timm
from torchvision import transforms

from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneOut
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [2]:
TRAIN_CSV  = '/kaggle/input/competitions/soil-grain-size-from-photos/Training_labels_updated.csv'
PPM_CSV    = '/kaggle/input/competitions/soil-grain-size-from-photos/ppm_updated.csv'
SUB_CSV    = '/kaggle/input/competitions/soil-grain-size-from-photos/sample_submission.csv'
TRAIN_IMG_DIR = '/kaggle/input/competitions/soil-grain-size-from-photos/Training-All_Photos_updated'
TEST_IMG_DIR  = '/kaggle/input/competitions/soil-grain-size-from-photos/Test_All_Photos/Test_All_Photos'

DIAMS = np.array([0.002,0.0063,0.02,0.063,0.2,0.63,2.0,6.3,20.0,63.0,200.0])
COLS  = [str(d) if d not in [2.0,6.3,20.0,63.0] else str(int(d)) if d==2.0 or d==20.0 or d==63.0 else '6.3' for d in DIAMS]
# ใช้ header ตรงจาก sample_submission.csv แทนดีกว่า กัน mismatch
sub_sample = pd.read_csv(SUB_CSV)
COLS = list(sub_sample.columns[1:])
print(COLS)

['0.002', '0.0063', '0.02', '0.063', '0.2', '0.63', '2', '6.3', '20', '63', '200']


In [3]:
train_df = pd.read_csv(TRAIN_CSV)
ppm_df   = pd.read_csv(PPM_CSV)

test_df = sub_sample[['sample_id']].copy()   # รายชื่อ test sample มาจาก sample_submission.csv

print(train_df.shape, test_df.shape)
print(ppm_df.head())

(24, 12) (10, 1)
                     phone                   camera  width  height     ppm
0                iPhone 14                iPhone 14   4032    3024  13.942
1                iPhone 16                iPhone 16   5712    4284  19.525
2            Motorola Edge         motorola edge 20   4000    1800  11.492
3  Motorola Edge 60 Fusion  Motorola Edge 60 fusion   4096    2304  12.465
4              Samsung A52                 SM-A525F   9248    6936  26.330


In [4]:
import glob, os

# 1) เช็คว่า path โฟลเดอร์ถูกไหม มีอะไรอยู่ข้างในบ้าง
print(os.path.exists(TRAIN_IMG_DIR))
print(os.listdir(TRAIN_IMG_DIR)[:20])

True
['Training-All_Photos_updated']


In [5]:
def index_images(img_dir, valid_sample_ids, camera_names):
    paths = glob.glob(f'{img_dir}/**/*.*', recursive=True)
    paths = [p for p in paths if p.lower().endswith(('.jpg','.jpeg','.png','.tif','.tiff'))]

    rows = []
    for p in paths:
        fname = os.path.basename(p)
        # sample_id เช่น F827, G190, H030 -> ตัวอักษร 1 ตัว + เลข 3 หลัก
        m = re.search(r'([A-Z]\d{3})', fname)
        sid = m.group(1) if m else None
        if sid not in valid_sample_ids:
            continue
        # หา camera name จาก ppm.csv โดยเช็คว่าชื่อกล้องปรากฏใน filename หรือไม่
        cam = next((c for c in camera_names if c.lower() in fname.lower()), None)
        rows.append({'sample_id': sid, 'camera': cam, 'path': p})
    return pd.DataFrame(rows)

camera_col = [c for c in ppm_df.columns if 'camera' in c.lower()][0]
camera_names = ppm_df[camera_col].astype(str).unique().tolist()

train_img_df = index_images(TRAIN_IMG_DIR, set(train_df['sample_id']), camera_names)
test_img_df  = index_images(TEST_IMG_DIR,  set(test_df['sample_id']),  camera_names)

print(train_img_df.shape, test_img_df.shape)
print(train_img_df['camera'].isna().sum(), 'images ที่ระบุกล้องไม่ได้ -> ต้องปรับ regex')
train_img_df.head()

(127, 3) (0, 0)
127 images ที่ระบุกล้องไม่ได้ -> ต้องปรับ regex


,sample_id,camera,path
0,H666,None,/kaggle/input/competitions/soil-grain-size-fro...
1,H374,None,/kaggle/input/competitions/soil-grain-size-fro...
2,H374,None,/kaggle/input/competitions/soil-grain-size-fro...
3,H368,None,/kaggle/input/competitions/soil-grain-size-fro...
4,H038,None,/kaggle/input/competitions/soil-grain-size-fro...


In [6]:
print(os.path.exists(TEST_IMG_DIR))
print(os.listdir(TEST_IMG_DIR)[:20])

True
['iPhone14_HPC_Münster_BS6_9,0-10m (3).JPG', 'iPhone16_HPC_Kleinkummerfeld 18-3 (1).JPG', 'iPhone16_HPC_Kleinkummerfeld 9-4 (2).JPG', 'iPhone16_HPC_Airbus BS6-3 (2).JPG', 'iPhone14_HPC_Münster_BS6_9,0-10m (2).JPG', 'iPhone_16_HPC_Airbus BS10-4bis7 (4).JPG', 'iPhone16_HPC_Airbus BS10-4bis7 (3).JPG', 'iPhone16_HPC_Kleinkummerfeld 2-2 (3).JPG', 'iPhone14_HPC_Münster_BS6_9,0-10m (1).JPG', 'iPhone16_HPC_Kleinkummerfeld 18-3 (2).JPG', 'iPhone16_HPC_Kleinkummerfeld 2-2 (4).JPG', 'iPhone16_HPC_Kleinkummerfeld 2-3 (3).JPG', 'iPhone14_HPC_Audorfring (3).JPG', 'iPhone16_HPC_Kleinkummerfeld 2-2 (2).JPG', 'iPhone16_HPC_Kleinkummerfeld 18-3 (4).JPG', 'iPhone16_HPC_Airbus BS12-5bis8 (3).JPG', 'iPhone14_HPC_Testfeld Lidl WHV (1).JPG', 'iPhone16_HPC_Kleinkummerfeld 2-3 (2).JPG', 'iPhone14_HPC_Audorfring (1).JPG', 'iPhone14_HPC_Testfeld Lidl WHV (3).JPG']


In [6]:
print(train_df['sample_id'].unique())
print(test_df['sample_id'].unique())
print(ppm_df.to_string())   # อยากเห็น camera column เต็ม ๆ ว่าชื่อกล้องเขียนแบบไหน (มี underscore/space ไหม)

['F827' 'G190' 'H030' 'H037' 'H038' 'H126' 'H181' 'H183' 'H366' 'H367'
 'H368' 'H371' 'H372' 'H374' 'H405' 'H493' 'H516' 'H549' 'H615' 'H616'
 'H617' 'H637' 'H666' 'H668']
['HPC_Airbus BS6-3' 'HPC_Audorfring' 'HPC_Muenster_BS6_9_0-10m'
 'HPC_Airbus BS10-4bis7' 'HPC_Airbus BS12-5bis8' 'HPC_Kleinkummerfeld 2-2'
 'HPC_Kleinkummerfeld 2-3' 'HPC_Kleinkummerfeld 9-4'
 'HPC_Kleinkummerfeld 18-3' 'HPC_Testfeld Lidl WHV']
                     phone                   camera  width  height     ppm
0                iPhone 14                iPhone 14   4032    3024  13.942
1                iPhone 16                iPhone 16   5712    4284  19.525
2            Motorola Edge         motorola edge 20   4000    1800  11.492
3  Motorola Edge 60 Fusion  Motorola Edge 60 fusion   4096    2304  12.465
4              Samsung A52                 SM-A525F   9248    6936  26.330


In [ ]:
print(os.listdir(TRAIN_IMG_DIR)[:20])

In [7]:
sub = os.path.join(TRAIN_IMG_DIR, 'Training-All_Photos_updated')
print(os.listdir(sub)[:20])

['Samsung_A52_H666_01.jpg', 'Motorola_Edge_60_fusion_H374_02.jpg', 'Motorola_Edge_60_fusion_H374_03.jpg', 'Motorola_Edge_H368_04.jpg', 'Samsung_A52_H038_01.jpg', 'Samsung_A52_H637_03.jpg', 'Motorola_Edge_H038_02.jpg', 'Motorola_Edge_H366_01.jpg', 'Samsung_A52_H405_01.jpg', 'Motorola_Edge_H666_03.jpg', 'Motorola_Edge_H037_03.jpg', 'Motorola_Edge_H668_02.jpg', 'Samsung_A52_H615_01.jpg', 'Motorola_Edge_H493_01.jpg', 'Samsung_A52_F827_03.jpg', 'Motorola_Edge_H367_02.jpg', 'Motorola_Edge_F827_02.jpg', 'Samsung_A52_G190_03.jpg', 'Motorola_Edge_H405_03.jpg', 'Samsung_A52_H371_01.jpg']


In [7]:
def normalize(s):
    s = str(s).lower()
    repl = {'ü':'ue','ö':'oe','ä':'ae','ß':'ss'}
    for a,b in repl.items():
        s = s.replace(a,b)
    return re.sub(r'[^a-z0-9]', '', s)

# camera alias: normalize + เรียงยาว->สั้น กัน "Motorola Edge" match ก่อน "Motorola Edge 60 Fusion"
camera_alias_map = []
for _, row in ppm_df.iterrows():
    for col in ['phone', 'camera']:
        if col in ppm_df.columns:
            camera_alias_map.append((normalize(row[col]), row[col]))
camera_alias_map = sorted(set(camera_alias_map), key=lambda x: len(x[0]), reverse=True)

def index_images(img_dir, valid_sample_ids, camera_alias_map, id_mode='auto'):
    paths = glob.glob(f'{img_dir}/**/*.*', recursive=True)
    paths = [p for p in paths if p.lower().endswith(('.jpg','.jpeg','.png','.tif','.tiff'))]

    ids_norm = sorted([(normalize(s), s) for s in valid_sample_ids],
                       key=lambda x: len(x[0]), reverse=True)

    rows, unmatched = [], []
    for p in paths:
        fname = os.path.basename(p)
        fname_norm = normalize(fname)

        # 1) ลองใช้ regex F827-style ก่อน (เร็ว/แม่นสำหรับ train)
        m = re.search(r'\b([A-Z]\d{3})\b', fname)
        sid = m.group(1) if (m and m.group(1) in valid_sample_ids) else None

        # 2) ถ้าไม่เจอ ลอง substring match แบบ normalize (สำหรับ test ที่เป็นชื่อสถานที่)
        if sid is None:
            sid = next((orig for norm, orig in ids_norm if norm in fname_norm), None)

        if sid is None:
            unmatched.append(fname)
            continue

        cam = next((cam_orig for alias_norm, cam_orig in camera_alias_map if alias_norm in fname_norm), None)
        rows.append({'sample_id': sid, 'camera': cam, 'path': p})

    print(f'{len(unmatched)}/{len(paths)} ไฟล์ match sample_id ไม่ได้ ตัวอย่าง:', unmatched[:8])
    return pd.DataFrame(rows)

def get_ppm(camera_name):
    if camera_name is None:
        return np.nan
    match = ppm_df[(ppm_df['phone']==camera_name) | (ppm_df['camera']==camera_name)]
    return match['ppm'].values[0] if len(match) else np.nan

train_img_df = index_images(TRAIN_IMG_DIR, set(train_df['sample_id']), camera_alias_map)
test_img_df  = index_images(TEST_IMG_DIR,  set(test_df['sample_id']),  camera_alias_map)

train_img_df['ppm'] = train_img_df['camera'].map(get_ppm)
test_img_df['ppm']  = test_img_df['camera'].map(get_ppm)

print('train:', train_img_df.shape, 'unique samples:', train_img_df['sample_id'].nunique(), '/', train_df['sample_id'].nunique())
print('test :', test_img_df.shape, 'unique samples:', test_img_df['sample_id'].nunique(), '/', test_df['sample_id'].nunique())
print('missing ppm (train/test):', train_img_df['ppm'].isna().sum(), test_img_df['ppm'].isna().sum())

3/130 ไฟล์ match sample_id ไม่ได้ ตัวอย่าง: ['Samsung_A52_H031_01.jpg', 'Samsung_A52_H031_02.jpg', 'Samsung_A52_H031_03.jpg']
0/35 ไฟล์ match sample_id ไม่ได้ ตัวอย่าง: []
train: (127, 4) unique samples: 24 / 24
test : (35, 4) unique samples: 10 / 10
missing ppm (train/test): 0 0


In [8]:
backbone = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
backbone.eval().to(device)

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

@torch.no_grad()
def deep_features(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    x = preprocess(img_rgb).unsqueeze(0).to(device)
    feat = backbone(x).cpu().numpy().flatten()
    return feat

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [9]:
def grain_spread_features(img_bgr, ppm):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray_small = cv2.resize(gray, (512, 512))
    _, th = cv2.threshold(gray_small, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(th, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    areas_px = np.array([cv2.contourArea(c) for c in contours if cv2.contourArea(c) > 4])

    if len(areas_px) > 2 and ppm and ppm > 0:
        diam_px = 2 * np.sqrt(areas_px / np.pi)
        diam_mm = diam_px / ppm
        mean_d, std_d = diam_mm.mean(), diam_mm.std()
        cv_d = std_d / mean_d if mean_d > 0 else 0
        p25, p75 = np.percentile(diam_mm, [25, 75])
        iqr_ratio = (p75 - p25) / mean_d if mean_d > 0 else 0
        skew = ((diam_mm - mean_d) ** 3).mean() / (std_d ** 3 + 1e-9)
        max_p25_ratio = diam_mm.max() / (p25 + 1e-9)
        return np.array([cv_d, iqr_ratio, skew, max_p25_ratio])
    return np.zeros(4)

In [10]:
def extract_spread(img_df):
    feats = []
    for _, row in tqdm(img_df.iterrows(), total=len(img_df)):
        img = cv2.imread(row['path'])
        feats.append(grain_spread_features(img, row['ppm']))
    return np.vstack(feats)

train_spread = extract_spread(train_img_df)
test_spread  = extract_spread(test_img_df)

NameError: name 'feats' is not defined

In [11]:
def inner_loocv_score(X_inner, y_clr_inner, y_cum_inner, builder, kwargs):
    loo = LeaveOneOut()
    preds = np.zeros_like(y_cum_inner)
    for tr, va in loo.split(X_inner):
        pipe = builder(**kwargs)
        pipe.fit(X_inner[tr], y_clr_inner[tr])
        pred_clr = pipe.predict(X_inner[va])
        pred_bins = clr_inverse(pred_clr)
        pred_cum = np.cumsum(pred_bins, axis=1)
        pred_cum = np.clip(pred_cum, 0, 100)
        pred_cum[:, -1] = 100.0
        preds[va] = pred_cum
    return emd_score(y_cum_inner, preds).mean()

outer_loo = LeaveOneOut()
n_samples = len(y_true_cum)
nested_preds = np.zeros_like(y_true_cum)
selected_configs = []

for outer_tr, outer_va in outer_loo.split(np.arange(n_samples)):
    y_clr_inner = y_clr[outer_tr]
    y_cum_inner = y_true_cum[outer_tr]

    # --- inner search: เลือก config ที่ดีที่สุด เห็นแค่ 23 samples (ไม่เห็น outer_va เลย) ---
    best_inner = (np.inf, None)
    for name, builder, source, kwargs in candidates_aug:          # <-- เปลี่ยนจาก candidates
        X_inner = sources_aug[source][outer_tr]                   # <-- เปลี่ยนจาก sources
        s = inner_loocv_score(X_inner, y_clr_inner, y_cum_inner, builder, kwargs)
        if s < best_inner[0]:
            best_inner = (s, (name, builder, source, kwargs))

    _, (name, builder, source, kwargs) = best_inner
    selected_configs.append(name)

    # --- เทรน config ที่ถูกเลือกด้วย 23 samples เต็ม ๆ แล้วทำนาย sample ที่ถูก hold-out ---
    Xs = sources_aug[source]                                       # <-- เปลี่ยนจาก sources
    pipe = builder(**kwargs)
    pipe.fit(Xs[outer_tr], y_clr[outer_tr])
    pred_clr = pipe.predict(Xs[outer_va])
    pred_bins = clr_inverse(pred_clr)
    pred_cum = np.cumsum(pred_bins, axis=1)
    pred_cum = np.clip(pred_cum, 0, 100)
    pred_cum[:, -1] = 100.0
    nested_preds[outer_va] = pred_cum

nested_score = emd_score(y_true_cum, nested_preds).mean()
print('Nested CV EMD (ตัวเลขที่เชื่อถือได้จริง):', nested_score)

from collections import Counter
print('ความถี่ config ที่ถูกเลือกในแต่ละ outer fold:', Counter(selected_configs))

NameError: name 'y_true_cum' is not defined

In [12]:
def classical_features(img_bgr, ppm):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray_small = cv2.resize(gray, (512,512))

    # --- texture: GLCM ---
    glcm = graycomatrix(gray_small, distances=[1,3], angles=[0, np.pi/4],
                         levels=256, symmetric=True, normed=True)
    glcm_feats = np.hstack([
        graycoprops(glcm, prop).flatten()
        for prop in ['contrast','homogeneity','energy','correlation']
    ])

    # --- LBP histogram ---
    lbp = local_binary_pattern(gray_small, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp, bins=10, range=(0,10), density=True)

    # --- color histogram ---
    hsv = cv2.cvtColor(cv2.resize(img_bgr,(512,512)), cv2.COLOR_BGR2HSV)
    color_hist = np.hstack([
        np.histogram(hsv[...,i], bins=16, range=(0,256), density=True)[0]
        for i in range(3)
    ])

    # --- apparent grain-size proxy: segment ด้วย Otsu + edge, แปลง pixel->mm ด้วย ppm ---
    edges = cv2.Canny(gray_small, 50, 150)
    edge_density = edges.mean() / 255.0

    _, th = cv2.threshold(gray_small, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(th, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    areas_px = np.array([cv2.contourArea(c) for c in contours if cv2.contourArea(c) > 4])
    if len(areas_px) > 0 and ppm and ppm > 0:
        diam_px = 2*np.sqrt(areas_px/np.pi)
        diam_mm = diam_px / ppm   # ppm = pixels per mm
        size_stats = np.array([
            diam_mm.mean(), diam_mm.std(), np.median(diam_mm),
            np.percentile(diam_mm,25), np.percentile(diam_mm,75),
            diam_mm.max(), len(diam_mm)
        ])
    else:
        size_stats = np.zeros(7)

    return np.hstack([glcm_feats, lbp_hist, color_hist, edge_density, size_stats])

In [13]:
def extract_all(img_df):
    deep_list, cls_list = [], []
    for _, row in tqdm(img_df.iterrows(), total=len(img_df)):
        img = cv2.imread(row['path'])
        if img is None:
            continue
        deep_list.append(deep_features(img))
        cls_list.append(classical_features(img, row['ppm']))
    deep_arr = np.vstack(deep_list)
    cls_arr  = np.vstack(cls_list)
    return deep_arr, cls_arr

train_deep, train_cls = extract_all(train_img_df)
test_deep,  test_cls  = extract_all(test_img_df)

train_feats = np.hstack([train_deep, train_cls])
test_feats  = np.hstack([test_deep, test_cls])

train_img_df = train_img_df.reset_index(drop=True)
test_img_df  = test_img_df.reset_index(drop=True)

  0%|          | 0/127 [00:00<?, ?it/s]

  0%|          | 0/35 [00:00<?, ?it/s]

In [17]:
import numpy as np
import pandas as pd

DATA_DIR   = '/kaggle/input/competitions/soil-grain-size-from-photos'
TRAIN_CSV  = f'{DATA_DIR}/Training_labels_updated.csv'
SUB_CSV    = f'{DATA_DIR}/sample_submission.csv'

train_df = pd.read_csv(TRAIN_CSV)
sub_sample = pd.read_csv(SUB_CSV)
COLS = list(sub_sample.columns[1:])

DIAMS = np.array([0.002,0.0063,0.02,0.063,0.2,0.63,2.0,6.3,20.0,63.0,200.0])
y_true_cum = train_df[COLS].values.astype(float)

log_d = np.log10(DIAMS)
def emd_score(y_true, y_pred):
    diff = np.abs(y_true - y_pred)
    w = np.diff(log_d)
    return np.sum((diff[:, :-1] + diff[:, 1:]) / 2 * w, axis=1)

In [18]:
from scipy.optimize import curve_fit

def weibull_cdf(d, lam, k):
    return 100 * (1 - np.exp(-(d/lam)**k))

oracle_preds = np.zeros_like(y_true_cum)
fit_params = []
for i in range(len(y_true_cum)):
    try:
        popt, _ = curve_fit(weibull_cdf, DIAMS, y_true_cum[i], p0=[1.0, 1.0],
                             bounds=([1e-6, 1e-6], [1e6, 10]), maxfev=5000)
        oracle_preds[i] = weibull_cdf(DIAMS, *popt)
        fit_params.append(popt)
    except Exception as e:
        oracle_preds[i] = y_true_cum[i]
        fit_params.append((np.nan, np.nan))

oracle_preds = np.clip(oracle_preds, 0, 100)
oracle_preds[:, -1] = 100.0
print('Oracle EMD:', emd_score(y_true_cum, oracle_preds).mean())

Oracle EMD: 7.932974723528012


In [19]:
from scipy.optimize import curve_fit

def weibull_cdf(d, lam, k):
    return 100 * (1 - np.exp(-(d/lam)**k))

oracle_preds = np.zeros_like(y_true_cum)
fit_params = []
for i in range(len(y_true_cum)):
    try:
        popt, _ = curve_fit(weibull_cdf, DIAMS, y_true_cum[i], p0=[1.0, 1.0],
                             bounds=([1e-6, 1e-6], [1e6, 10]), maxfev=5000)
        oracle_preds[i] = weibull_cdf(DIAMS, *popt)
        fit_params.append(popt)
    except Exception as e:
        oracle_preds[i] = y_true_cum[i]  # fit ไม่ผ่าน fallback ใช้ true value
        fit_params.append((np.nan, np.nan))

oracle_preds = np.clip(oracle_preds, 0, 100)
oracle_preds[:, -1] = 100.0
print('Oracle EMD (best-case ถ้าโมเดลทำนาย Weibull param ได้แม่นสมบูรณ์แบบ):',
      emd_score(y_true_cum, oracle_preds).mean())

Oracle EMD (best-case ถ้าโมเดลทำนาย Weibull param ได้แม่นสมบูรณ์แบบ): 7.932974723528012


In [20]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneOut

def build_pca_ridge_original(n_comp=5, alpha=100):
    return Pipeline([('pca', PCA(n_components=n_comp, random_state=0)), ('ridge', Ridge(alpha=alpha))])

# แปลง (lambda, k) เป็น log-space เพื่อให้ regression ทำงานได้ดีขึ้น (ค่าเป็นบวกเสมอ)
y_weibull = np.array(fit_params)          # shape (24, 2) = [lambda, k]
y_weibull_log = np.log(y_weibull)         # log(lambda), log(k)

def weibull_loocv(X, n_comp, alpha):
    loo = LeaveOneOut()
    preds_cum = np.zeros_like(y_true_cum)
    for tr_idx, va_idx in loo.split(X):
        pipe = build_pca_ridge_original(n_comp=n_comp, alpha=alpha)
        pipe.fit(X[tr_idx], y_weibull_log[tr_idx])
        pred_log = pipe.predict(X[va_idx])
        pred_params = np.exp(pred_log)   # กลับเป็น lambda, k จริง
        for j, va_i in enumerate(va_idx):
            lam, k = pred_params[j]
            curve = weibull_cdf(DIAMS, lam, k)
            preds_cum[va_i] = np.clip(curve, 0, 100)
    preds_cum[:, -1] = 100.0
    return emd_score(y_true_cum, preds_cum).mean()

best_score, best_cfg = np.inf, None
for n_comp in [3, 5, 8, 12]:
    for alpha in [1, 10, 50, 100, 300]:
        s = weibull_loocv(X_full, n_comp, alpha)
        if s < best_score:
            best_score, best_cfg = s, (n_comp, alpha)

print('best Weibull-target LOOCV EMD:', best_score, 'cfg:', best_cfg)

NameError: name 'X_full' is not defined

In [21]:
def aggregate_only(img_df, feats):
    df = pd.DataFrame(feats)
    df['sample_id'] = img_df['sample_id'].values
    mean = df.groupby('sample_id').mean()
    std  = df.groupby('sample_id').std().fillna(0)
    std.columns = [f'std_{c}' for c in std.columns]
    return pd.concat([mean, std], axis=1)

train_deep_agg  = aggregate_only(train_img_df, train_deep).reindex(train_df['sample_id'])
train_cls_agg   = aggregate_only(train_img_df, train_cls).reindex(train_df['sample_id'])
train_spread_agg = aggregate_only(train_img_df, train_spread).reindex(train_df['sample_id'])
test_spread_agg  = aggregate_only(test_img_df,  test_spread).reindex(test_df['sample_id'])

X_full = np.hstack([train_deep_agg.values, train_cls_agg.values])
X_full_aug = np.hstack([X_full, train_spread_agg.values])
X_combo = X_full  # เดียวกับที่ใช้ก่อนหน้า

y_true_cum = train_df[COLS].values.astype(float)

EPS = 1e-3
def cum_to_bins(cum):
    bins = np.diff(np.hstack([np.zeros((cum.shape[0],1)), cum]), axis=1)
    return np.clip(bins, EPS, None)
def clr_transform(bins):
    b = bins / bins.sum(axis=1, keepdims=True) * 100
    log_b = np.log(b)
    return log_b - log_b.mean(axis=1, keepdims=True)
def clr_inverse(clr, total=100.0):
    b = np.exp(clr)
    return b / b.sum(axis=1, keepdims=True) * total

y_clr = clr_transform(cum_to_bins(y_true_cum))

log_d = np.log10(DIAMS)
def emd_score(y_true, y_pred):
    diff = np.abs(y_true - y_pred)
    w = np.diff(log_d)
    return np.sum((diff[:, :-1] + diff[:, 1:]) / 2 * w, axis=1)

NameError: name 'train_spread' is not defined

In [16]:
def build_classical_ridge(alpha=10.0):
    return Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=alpha))])

def build_deep_pca_classical(n_comp=10, alpha=10.0):
    n_deep = train_deep_agg.shape[1]
    deep_idx = list(range(n_deep))
    cls_idx  = list(range(n_deep, X_combo.shape[1]))
    ct = ColumnTransformer([
        ('deep_pca', Pipeline([('scale', StandardScaler()), ('pca', PCA(n_components=n_comp, random_state=0))]), deep_idx),
        ('cls_scale', StandardScaler(), cls_idx),
    ])
    return Pipeline([('features', ct), ('ridge', Ridge(alpha=alpha))])

def build_pca_ridge_original(n_comp=5, alpha=100):
    return Pipeline([('pca', PCA(n_components=n_comp, random_state=0)), ('ridge', Ridge(alpha=alpha))])

sources_aug = {
    'cls': train_cls_agg.values,
    'combo': X_combo,
    'full': X_full,
    'full_aug': X_full_aug,
}

candidates_aug = [
    ('classical_a10',   build_classical_ridge,   'cls',   dict(alpha=10)),
    ('classical_a100',  build_classical_ridge,   'cls',   dict(alpha=100)),
    ('classical_a300',  build_classical_ridge,   'cls',   dict(alpha=300)),
    ('combo_nc5_a100',  build_deep_pca_classical, 'combo', dict(n_comp=5, alpha=100)),
    ('combo_nc5_a3000', build_deep_pca_classical, 'combo', dict(n_comp=5, alpha=3000)),
    ('full_pca5_a100',  build_pca_ridge_original, 'full',  dict(n_comp=5, alpha=100)),
    ('full_aug_pca5_a100', build_pca_ridge_original, 'full_aug', dict(n_comp=5, alpha=100)),
    ('full_aug_pca8_a100', build_pca_ridge_original, 'full_aug', dict(n_comp=8, alpha=100)),
]

NameError: name 'X_combo' is not defined

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneOut

In [15]:
def inner_loocv_score(X_inner, y_clr_inner, y_cum_inner, builder, kwargs):
    loo = LeaveOneOut()
    preds = np.zeros_like(y_cum_inner)
    for tr, va in loo.split(X_inner):
        pipe = builder(**kwargs)
        pipe.fit(X_inner[tr], y_clr_inner[tr])
        pred_clr = pipe.predict(X_inner[va])
        pred_bins = clr_inverse(pred_clr)
        pred_cum = np.cumsum(pred_bins, axis=1)
        pred_cum = np.clip(pred_cum, 0, 100)
        pred_cum[:, -1] = 100.0
        preds[va] = pred_cum
    return emd_score(y_cum_inner, preds).mean()

outer_loo = LeaveOneOut()
n_samples = len(y_true_cum)
nested_preds = np.zeros_like(y_true_cum)
selected_configs = []

for outer_tr, outer_va in outer_loo.split(np.arange(n_samples)):
    y_clr_inner = y_clr[outer_tr]
    y_cum_inner = y_true_cum[outer_tr]

    # --- inner search: เลือก config ที่ดีที่สุด เห็นแค่ 23 samples (ไม่เห็น outer_va เลย) ---
    best_inner = (np.inf, None)
    for name, builder, source, kwargs in candidates_aug:
        X_inner = sources_aug[source][outer_tr]
        s = inner_loocv_score(X_inner, y_clr_inner, y_cum_inner, builder, kwargs)
        if s < best_inner[0]:
            best_inner = (s, (name, builder, source, kwargs))

    _, (name, builder, source, kwargs) = best_inner
    selected_configs.append(name)

    # --- เทรน config ที่ถูกเลือกด้วย 23 samples เต็ม ๆ แล้วทำนาย sample ที่ถูก hold-out ---
    Xs = sources_aug[source]
    pipe = builder(**kwargs)
    pipe.fit(Xs[outer_tr], y_clr[outer_tr])
    pred_clr = pipe.predict(Xs[outer_va])
    pred_bins = clr_inverse(pred_clr)
    pred_cum = np.cumsum(pred_bins, axis=1)
    pred_cum = np.clip(pred_cum, 0, 100)
    pred_cum[:, -1] = 100.0
    nested_preds[outer_va] = pred_cum

nested_score = emd_score(y_true_cum, nested_preds).mean()
print('Nested CV EMD (ตัวเลขที่เชื่อถือได้จริง):', nested_score)

from collections import Counter
print('ความถี่ config ที่ถูกเลือกในแต่ละ outer fold:', Counter(selected_configs))

Nested CV EMD (ตัวเลขที่เชื่อถือได้จริง): 39.519875768765175
ความถี่ config ที่ถูกเลือกในแต่ละ outer fold: Counter({'combo_nc5_a3000': 24})


In [17]:
candidates_safe = [
    ('full_pca3_a100',     build_pca_ridge_original, 'full',     dict(n_comp=3, alpha=100)),
    ('full_pca5_a100',     build_pca_ridge_original, 'full',     dict(n_comp=5, alpha=100)),
    ('full_pca8_a100',     build_pca_ridge_original, 'full',     dict(n_comp=8, alpha=100)),
    ('full_aug_pca3_a100', build_pca_ridge_original, 'full_aug', dict(n_comp=3, alpha=100)),
    ('full_aug_pca5_a100', build_pca_ridge_original, 'full_aug', dict(n_comp=5, alpha=100)),
    ('full_aug_pca8_a100', build_pca_ridge_original, 'full_aug', dict(n_comp=8, alpha=100)),
]

In [18]:
for outer_tr, outer_va in outer_loo.split(np.arange(n_samples)):
    y_clr_inner = y_clr[outer_tr]
    y_cum_inner = y_true_cum[outer_tr]

    best_inner = (np.inf, None)
    for name, builder, source, kwargs in candidates_safe:      # <-- เปลี่ยนตรงนี้
        X_inner = sources_aug[source][outer_tr]
        s = inner_loocv_score(X_inner, y_clr_inner, y_cum_inner, builder, kwargs)
        if s < best_inner[0]:
            best_inner = (s, (name, builder, source, kwargs))

    _, (name, builder, source, kwargs) = best_inner
    selected_configs.append(name)

    Xs = sources_aug[source]
    pipe = builder(**kwargs)
    pipe.fit(Xs[outer_tr], y_clr[outer_tr])
    pred_clr = pipe.predict(Xs[outer_va])
    pred_bins = clr_inverse(pred_clr)
    pred_cum = np.cumsum(pred_bins, axis=1)
    pred_cum = np.clip(pred_cum, 0, 100)
    pred_cum[:, -1] = 100.0
    nested_preds[outer_va] = pred_cum

nested_score = emd_score(y_true_cum, nested_preds).mean()
print('Nested CV EMD (safe-only):', nested_score)
print('Counter:', Counter(selected_configs))

Nested CV EMD (safe-only): 45.32295953933262
Counter: Counter({'combo_nc5_a3000': 24, 'full_aug_pca3_a100': 21, 'full_pca3_a100': 3})


In [21]:
final_pipe = build_pca_ridge_original(n_comp=3, alpha=100)
final_pipe.fit(X_full_aug, y_clr)

test_deep_agg = aggregate_only(test_img_df, test_deep).reindex(test_df['sample_id'])
test_cls_agg  = aggregate_only(test_img_df, test_cls).reindex(test_df['sample_id'])
test_agg = np.hstack([test_deep_agg.values, test_cls_agg.values])


X_test_full_aug = np.hstack([test_agg, test_spread_agg.values])
# ระวัง: test_agg ต้องมาจากชุดเดียวกับ train_agg (deep+cls รวมกัน) เช็คว่า column order ตรงกับ X_full ก่อน concat test_spread_agg


pred_clr_test = final_pipe.predict(X_test_full_aug)
pred_bins_test = clr_inverse(pred_clr_test)
pred_cum_test  = np.cumsum(pred_bins_test, axis=1)
pred_cum_test = np.maximum.accumulate(pred_cum_test, axis=1)
pred_cum_test = np.clip(pred_cum_test, 0, 100)
pred_cum_test[:, -1] = 100.0

submission = pd.DataFrame(pred_cum_test, columns=COLS)
submission.insert(0, 'sample_id', test_df['sample_id'].values)
assert (submission[COLS].diff(axis=1).iloc[:,1:] >= -1e-6).all().all()
assert np.allclose(submission[COLS[-1]], 100.0)
submission.to_csv('submission.csv', index=False)
submission.head()

,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,HPC_Airbus BS6-3,2.379459,7.507603,24.575893,49.911236,84.164425,96.701864,98.375576,99.956086,99.979657,99.981046,100.0
1,HPC_Audorfring,0.057452,0.069814,0.108568,3.847884,40.647744,82.658219,91.166103,99.869774,99.942243,99.942289,100.0
2,HPC_Muenster_BS6_9_0-10m,0.372276,4.120443,10.851040,16.990050,20.279284,25.704804,38.160623,70.580640,81.986919,99.995039,100.0
3,HPC_Airbus BS10-4bis7,2.236230,5.761982,18.625916,42.998049,84.692530,97.806316,98.989133,99.969123,99.977511,99.977755,100.0
4,HPC_Airbus BS12-5bis8,2.525866,10.352253,34.440397,59.412556,85.248491,95.816300,97.804055,99.927578,99.978690,99.985158,100.0


In [23]:
X = train_agg.values
loo = LeaveOneOut()

best_score, best_cfg = np.inf, None
for n_comp in [5, 8, 12, 16]:
    for alpha in [1.0, 5.0, 10.0, 30.0, 100.0]:
        preds_cum = np.zeros_like(train_df[COLS].values, dtype=float)
        for tr_idx, va_idx in loo.split(X):
            pca = PCA(n_components=n_comp, random_state=0).fit(X[tr_idx])
            Xtr, Xva = pca.transform(X[tr_idx]), pca.transform(X[va_idx])

            model = Ridge(alpha=alpha)
            model.fit(Xtr, y_clr[tr_idx])
            pred_clr = model.predict(Xva)

            pred_bins = clr_inverse(pred_clr)
            pred_cum = np.cumsum(pred_bins, axis=1)
            pred_cum = np.clip(pred_cum, 0, 100)
            pred_cum[:, -1] = 100.0
            preds_cum[va_idx] = pred_cum

        score = emd_score(train_df[COLS].values.astype(float), preds_cum).mean()
        if score < best_score:
            best_score, best_cfg = score, (n_comp, alpha)

print('best LOOCV EMD:', best_score, 'cfg:', best_cfg)

NameError: name 'train_agg' is not defined

In [ ]:
n_comp, alpha = best_cfg
pca_final = PCA(n_components=n_comp, random_state=0).fit(X)
X_pca = pca_final.transform(X)

model_final = Ridge(alpha=alpha)
model_final.fit(X_pca, y_clr)

In [ ]:
X_test_pca = pca_final.transform(test_agg.values)
pred_clr_test = model_final.predict(X_test_pca)

pred_bins_test = clr_inverse(pred_clr_test)
pred_cum_test  = np.cumsum(pred_bins_test, axis=1)

pred_cum_test = np.maximum.accumulate(pred_cum_test, axis=1)   # กัน monotonic เผื่อ float error
pred_cum_test = np.clip(pred_cum_test, 0, 100)
pred_cum_test[:, -1] = 100.0

In [ ]:
def build_pca_ridge_original(n_comp=5, alpha=100):
    return Pipeline([
        ('pca', PCA(n_components=n_comp, random_state=0)),   # ไม่มี StandardScaler
        ('ridge', Ridge(alpha=alpha)),
    ])

final_pipe = build_pca_ridge_original(n_comp=5, alpha=100)
final_pipe.fit(X_full, y_clr)

pred_clr_test = final_pipe.predict(test_agg.values)
pred_bins_test = clr_inverse(pred_clr_test)
pred_cum_test  = np.cumsum(pred_bins_test, axis=1)
pred_cum_test = np.maximum.accumulate(pred_cum_test, axis=1)
pred_cum_test = np.clip(pred_cum_test, 0, 100)
pred_cum_test[:, -1] = 100.0

In [ ]:
submission = pd.DataFrame(pred_cum_test, columns=COLS)
submission.insert(0, 'sample_id', test_df['sample_id'].values)

assert (submission[COLS].diff(axis=1).iloc[:,1:] >= -1e-6).all().all(), 'monotonic check failed'
assert np.allclose(submission[COLS[-1]], 100.0), 'last column must be 100'

submission.to_csv('submission.csv', index=False)
submission.head()

camera ppm

In [22]:
from PIL import Image

ppm_lookup    = ppm_df.set_index(['width', 'height'])['ppm'].to_dict()
camera_lookup = ppm_df.set_index(['width', 'height'])['camera'].to_dict()
global_mean_ppm = ppm_df['ppm'].mean()

def get_dims_ppm_camera(path):
    try:
        with Image.open(path) as im:
            w, h = im.size
    except Exception:
        return global_mean_ppm, None
    ppm = ppm_lookup.get((w, h), global_mean_ppm)
    cam = camera_lookup.get((w, h), None)
    return ppm, cam

In [23]:
def index_images(img_dir, valid_sample_ids):
    paths = glob.glob(f'{img_dir}/**/*.*', recursive=True)
    paths = [p for p in paths if p.lower().endswith(('.jpg','.jpeg','.png','.tif','.tiff'))]
    ids_norm = sorted([(normalize(s), s) for s in valid_sample_ids], key=lambda x: len(x[0]), reverse=True)

    rows = []
    for p in paths:
        fname = os.path.basename(p)
        fname_norm = normalize(fname)
        m = re.search(r'\b([A-Z]\d{3})\b', fname)
        sid = m.group(1) if (m and m.group(1) in valid_sample_ids) else None
        if sid is None:
            sid = next((orig for norm, orig in ids_norm if norm in fname_norm), None)
        if sid is None:
            continue
        rows.append({'sample_id': sid, 'path': p})
    return pd.DataFrame(rows)

train_img_df = index_images(TRAIN_IMG_DIR, set(train_df['sample_id']))
test_img_df  = index_images(TEST_IMG_DIR,  set(test_df['sample_id']))

train_img_df[['ppm','camera']] = train_img_df['path'].apply(lambda p: pd.Series(get_dims_ppm_camera(p)))
test_img_df[['ppm','camera']]  = test_img_df['path'].apply(lambda p: pd.Series(get_dims_ppm_camera(p)))

print(train_img_df.shape, test_img_df.shape)
print('sample_id ที่ match ไม่ได้ (train/test):',
      len(train_df) - train_img_df['sample_id'].nunique(),
      len(test_df) - test_img_df['sample_id'].nunique())
print('ppm fallback เป็นค่าเฉลี่ย (train/test):',
      (train_img_df['ppm'] == global_mean_ppm).sum(),
      (test_img_df['ppm'] == global_mean_ppm).sum())

(127, 4) (35, 4)
sample_id ที่ match ไม่ได้ (train/test): 0 0
ppm fallback เป็นค่าเฉลี่ย (train/test): 124 0


In [24]:
from PIL import Image

sample_paths = train_img_df['path'].iloc[:5].tolist()
for p in sample_paths:
    with Image.open(p) as im:
        print(os.path.basename(p), im.size)   # (width, height)

print(ppm_df[['phone','camera','width','height','ppm']])

Samsung_A52_H666_01.jpg (1599, 1200)
Motorola_Edge_60_fusion_H374_02.jpg (4096, 2304)
Motorola_Edge_60_fusion_H374_03.jpg (4096, 2304)
Motorola_Edge_H368_04.jpg (1600, 720)
Samsung_A52_H038_01.jpg (1599, 1200)
                     phone                   camera  width  height     ppm
0                iPhone 14                iPhone 14   4032    3024  13.942
1                iPhone 16                iPhone 16   5712    4284  19.525
2            Motorola Edge         motorola edge 20   4000    1800  11.492
3  Motorola Edge 60 Fusion  Motorola Edge 60 fusion   4096    2304  12.465
4              Samsung A52                 SM-A525F   9248    6936  26.330


รวม camera matching (ของเดิม) + คำนวณ scale จากขนาดจริง

In [25]:
def get_effective_ppm(path, camera_name):
    if camera_name is None:
        return global_mean_ppm
    row = ppm_df[(ppm_df['phone']==camera_name) | (ppm_df['camera']==camera_name)]
    if len(row) == 0:
        return global_mean_ppm
    native_w, native_h, native_ppm = row.iloc[0][['width','height','ppm']]
    try:
        with Image.open(path) as im:
            actual_w, actual_h = im.size
    except Exception:
        return native_ppm
    # เทียบแกนที่ยาวกว่ากับยาวกว่า กัน portrait/landscape สลับกัน
    scale = max(actual_w, actual_h) / max(native_w, native_h)
    return native_ppm * scale

In [27]:
def index_images(img_dir, valid_sample_ids, camera_alias_map):
    paths = glob.glob(f'{img_dir}/**/*.*', recursive=True)
    paths = [p for p in paths if p.lower().endswith(('.jpg','.jpeg','.png','.tif','.tiff'))]
    ids_norm = sorted([(normalize(s), s) for s in valid_sample_ids], key=lambda x: len(x[0]), reverse=True)

    rows = []
    for p in paths:
        fname = os.path.basename(p)
        fname_norm = normalize(fname)
        m = re.search(r'\b([A-Z]\d{3})\b', fname)
        sid = m.group(1) if (m and m.group(1) in valid_sample_ids) else None
        if sid is None:
            sid = next((orig for norm, orig in ids_norm if norm in fname_norm), None)
        if sid is None:
            continue
        cam = next((cam_orig for alias_norm, cam_orig in camera_alias_map if alias_norm in fname_norm), None)
        rows.append({'sample_id': sid, 'camera': cam, 'path': p})
    return pd.DataFrame(rows)

In [28]:
camera_alias_map = []
for _, row in ppm_df.iterrows():
    for col in ['phone', 'camera']:
        camera_alias_map.append((normalize(row[col]), row[col]))
camera_alias_map = sorted(set(camera_alias_map), key=lambda x: len(x[0]), reverse=True)

In [29]:
train_img_df = index_images(TRAIN_IMG_DIR, set(train_df['sample_id']), camera_alias_map)
test_img_df  = index_images(TEST_IMG_DIR,  set(test_df['sample_id']),  camera_alias_map)

train_img_df['ppm'] = train_img_df.apply(lambda r: get_effective_ppm(r['path'], r['camera']), axis=1)
test_img_df['ppm']  = test_img_df.apply(lambda r: get_effective_ppm(r['path'], r['camera']), axis=1)

print(train_img_df[['sample_id','camera','ppm']].head(10))

  sample_id                   camera        ppm
0      H666              Samsung A52   4.552516
1      H374  Motorola Edge 60 fusion  12.465000
2      H374  Motorola Edge 60 fusion  12.465000
3      H368            Motorola Edge   4.596800
4      H038              Samsung A52   4.552516
5      H637              Samsung A52   4.552516
6      H038            Motorola Edge   4.596800
7      H366            Motorola Edge   4.596800
8      H405              Samsung A52   4.552516
9      H666            Motorola Edge   4.596800


In [30]:
train_img_df = train_img_df.reset_index(drop=True)
test_img_df  = test_img_df.reset_index(drop=True)

def extract_all(img_df):
    deep_list, cls_list, spread_list = [], [], []
    for _, row in tqdm(img_df.iterrows(), total=len(img_df)):
        img = cv2.imread(row['path'])
        deep_list.append(deep_features(img))
        cls_list.append(classical_features(img, row['ppm']))
        spread_list.append(grain_spread_features(img, row['ppm']))
    return np.vstack(deep_list), np.vstack(cls_list), np.vstack(spread_list)

train_deep, train_cls, train_spread = extract_all(train_img_df)
test_deep,  test_cls,  test_spread  = extract_all(test_img_df)

  0%|          | 0/127 [00:00<?, ?it/s]

  0%|          | 0/35 [00:00<?, ?it/s]

In [31]:
train_deep_agg = aggregate_only(train_img_df, train_deep).reindex(train_df['sample_id'])
train_cls_agg  = aggregate_only(train_img_df, train_cls).reindex(train_df['sample_id'])
test_deep_agg  = aggregate_only(test_img_df, test_deep).reindex(test_df['sample_id'])
test_cls_agg   = aggregate_only(test_img_df, test_cls).reindex(test_df['sample_id'])

X_full = np.hstack([train_deep_agg.values, train_cls_agg.values])
test_agg = np.hstack([test_deep_agg.values, test_cls_agg.values])

y_true_cum = train_df[COLS].values.astype(float)
y_clr = clr_transform(cum_to_bins(y_true_cum))

In [32]:
loo = LeaveOneOut()
loocv_preds = np.zeros_like(y_true_cum)
for tr_idx, va_idx in loo.split(X_full):
    pipe = build_pca_ridge_original(n_comp=5, alpha=100)
    pipe.fit(X_full[tr_idx], y_clr[tr_idx])
    pred_clr = pipe.predict(X_full[va_idx])
    pred_bins = clr_inverse(pred_clr)
    pred_cum = np.cumsum(pred_bins, axis=1)
    pred_cum = np.clip(pred_cum, 0, 100)
    pred_cum[:, -1] = 100.0
    loocv_preds[va_idx] = pred_cum

print('LOOCV EMD (ppm แก้แล้ว):', emd_score(y_true_cum, loocv_preds).mean())

LOOCV EMD (ppm แก้แล้ว): 56.50525583525259


Ablation Study

Ablation Study = การทดลองแบบ "ตัดทอน/สลับองค์ประกอบทีละส่วน แล้ววัดผลเทียบกัน" เพื่อหาว่าส่วนไหนของ pipeline ที่ช่วยจริง ส่วนไหนแค่เพิ่ม noise — เป็นวิธีมาตรฐานในงานวิจัย ML เวลาสงสัยว่า "องค์ประกอบนี้จำเป็นไหม" แทนที่จะเดาแล้วเปลี่ยนทีเดียวทั้งระบบ

In [ ]:
def aggregate_only(img_df, feats):
    df = pd.DataFrame(feats)
    df['sample_id'] = img_df['sample_id'].values
    mean = df.groupby('sample_id').mean()
    std  = df.groupby('sample_id').std().fillna(0)
    std.columns = [f'std_{c}' for c in std.columns]
    return pd.concat([mean, std], axis=1)

train_deep_agg = aggregate_only(train_img_df, train_deep).reindex(train_df['sample_id'])
train_cls_agg  = aggregate_only(train_img_df, train_cls).reindex(train_df['sample_id'])

y_true_cum = train_df[COLS].values.astype(float)

print(train_deep_agg.shape, train_cls_agg.shape)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.compose import ColumnTransformer

def loocv_eval(X, pipeline_builder, **kwargs):
    loo = LeaveOneOut()
    preds_cum = np.zeros_like(y_true_cum)
    for tr_idx, va_idx in loo.split(X):
        pipe = pipeline_builder(**kwargs)
        pipe.fit(X[tr_idx], y_clr[tr_idx])
        pred_clr = pipe.predict(X[va_idx])
        pred_bins = clr_inverse(pred_clr)
        pred_cum = np.cumsum(pred_bins, axis=1)
        pred_cum = np.clip(pred_cum, 0, 100)
        pred_cum[:, -1] = 100.0
        preds_cum[va_idx] = pred_cum
    return emd_score(y_true_cum, preds_cum).mean()

In [ ]:
def build_classical_ridge(alpha=10.0):
    return Pipeline([
        ('scale', StandardScaler()),
        ('ridge', Ridge(alpha=alpha)),
    ])

best_cls = (np.inf, None)
for alpha in [100, 300, 1000, 3000, 10000]:
    s = loocv_eval(train_cls_agg.values, build_classical_ridge, alpha=alpha)
    if s < best_cls[0]:
        best_cls = (s, alpha)
print('classical-only Ridge (wider alpha):', best_cls)

In [ ]:
X_combo = np.hstack([train_deep_agg.values, train_cls_agg.values])
n_deep = train_deep_agg.shape[1]
deep_idx = list(range(n_deep))
cls_idx  = list(range(n_deep, X_combo.shape[1]))

def build_deep_pca_classical(n_comp=10, alpha=10.0):
    ct = ColumnTransformer([
        ('deep_pca', Pipeline([('scale', StandardScaler()),
                                ('pca', PCA(n_components=n_comp, random_state=0))]), deep_idx),
        ('cls_scale', StandardScaler(), cls_idx),
    ])
    return Pipeline([('features', ct), ('ridge', Ridge(alpha=alpha))])

best_combo = (np.inf, None)
for n_comp in [3, 5, 8]:
    for alpha in [100, 300, 1000, 3000]:
        s = loocv_eval(X_combo, build_deep_pca_classical, n_comp=n_comp, alpha=alpha)
        if s < best_combo[0]:
            best_combo = (s, (n_comp, alpha))
print('deep-PCA + classical Ridge (wider alpha):', best_combo)

ขยาย grid เฉพาะฝั่ง deep-PCA + classical ต่อ พร้อมทำละเอียดขึ้นรอบ ๆ 3000 เพื่อหาจุดต่ำสุดจริง:

In [ ]:
alphas_wide = [1000, 3000, 10000, 30000, 100000]

best_combo = (np.inf, None)
scores_log = []
for n_comp in [3, 5, 8]:
    for alpha in alphas_wide:
        s = loocv_eval(X_combo, build_deep_pca_classical, n_comp=n_comp, alpha=alpha)
        scores_log.append((n_comp, alpha, s))
        if s < best_combo[0]:
            best_combo = (s, (n_comp, alpha))

print('deep-PCA + classical Ridge (wider alpha):', best_combo)
for n_comp, alpha, s in scores_log:
    print(f'n_comp={n_comp}, alpha={alpha}: {s:.3f}')

In [ ]:
def univariate_score_func(X, y):
    X = np.asarray(X); y = np.asarray(y)
    Xc = X - X.mean(axis=0)
    yc = y - y.mean(axis=0)
    num = Xc.T @ yc
    denom = (np.sqrt((Xc**2).sum(axis=0))[:,None] * np.sqrt((yc**2).sum(axis=0))[None,:] + 1e-12)
    corr = num / denom
    score = np.max(np.abs(corr), axis=1)   # max |corr| ข้าม 11 target columns
    return score, np.zeros_like(score)      # pvalue หลอก ๆ ไม่ได้ใช้จริง

def build_selectk_ridge(k=50, alpha=10.0):
    return Pipeline([
        ('scale', StandardScaler()),
        ('select', SelectKBest(score_func=univariate_score_func, k=k)),
        ('ridge', Ridge(alpha=alpha)),
    ])

X_full = train_agg.values  # ชุดเต็ม 2724 มิติ (deep+classical, mean+std)

best_sel = (np.inf, None)
for k in [10, 20, 50, 100, 200]:
    for alpha in [1, 3, 10, 30, 100]:
        s = loocv_eval(X_full, build_selectk_ridge, k=k, alpha=alpha)
        if s < best_sel[0]:
            best_sel = (s, (k, alpha))

print('univariate-select + Ridge best EMD:', best_sel)

In [ ]:
def loocv_constant_baseline():
    n = len(y_true_cum)
    preds = np.zeros_like(y_true_cum)
    for i in range(n):
        mask = np.ones(n, bool); mask[i] = False
        preds[i] = y_true_cum[mask].mean(axis=0)
    return emd_score(y_true_cum, preds).mean()

print('LOOCV constant-mean baseline EMD:', loocv_constant_baseline())

ได้ interior minimum ที่แท้จริง

In [ ]:
test_deep_agg = aggregate_only(test_img_df, test_deep).reindex(test_df['sample_id'])
test_cls_agg  = aggregate_only(test_img_df, test_cls).reindex(test_df['sample_id'])
X_test_combo  = np.hstack([test_deep_agg.values, test_cls_agg.values])

In [ ]:
final_pipe = build_deep_pca_classical(n_comp=5, alpha=3000)
final_pipe.fit(X_combo, y_clr)

In [ ]:
pred_clr_test = final_pipe.predict(X_test_combo)
pred_bins_test = clr_inverse(pred_clr_test)
pred_cum_test  = np.cumsum(pred_bins_test, axis=1)

pred_cum_test = np.maximum.accumulate(pred_cum_test, axis=1)
pred_cum_test = np.clip(pred_cum_test, 0, 100)
pred_cum_test[:, -1] = 100.0

In [ ]:
submission = pd.DataFrame(pred_cum_test, columns=COLS)
submission.insert(0, 'sample_id', test_df['sample_id'].values)

assert (submission[COLS].diff(axis=1).iloc[:,1:] >= -1e-6).all().all(), 'monotonic check failed'
assert np.allclose(submission[COLS[-1]], 100.0), 'last column must be 100'

submission.to_csv('submission.csv', index=False)
submission.head()

score 106 ไม่ work

Nested CV

In [ ]:
def build_pca_ridge_full(n_comp=5, alpha=100):
    return Pipeline([
        ('scale', StandardScaler()),
        ('pca', PCA(n_components=n_comp, random_state=0)),
        ('ridge', Ridge(alpha=alpha)),
    ])

sources = {
    'cls':   train_cls_agg.values,
    'combo': X_combo,
    'full':  train_agg.values,
}

candidates = [
    ('classical_a10',   build_classical_ridge,   'cls',   dict(alpha=10)),
    ('classical_a100',  build_classical_ridge,   'cls',   dict(alpha=100)),
    ('classical_a300',  build_classical_ridge,   'cls',   dict(alpha=300)),
    ('combo_nc5_a100',  build_deep_pca_classical, 'combo', dict(n_comp=5, alpha=100)),
    ('combo_nc5_a1000', build_deep_pca_classical, 'combo', dict(n_comp=5, alpha=1000)),
    ('combo_nc5_a3000', build_deep_pca_classical, 'combo', dict(n_comp=5, alpha=3000)),
    ('combo_nc3_a3000', build_deep_pca_classical, 'combo', dict(n_comp=3, alpha=3000)),
    ('full_pca5_a100',  build_pca_ridge_full,     'full',  dict(n_comp=5, alpha=100)),
    ('full_pca8_a30',   build_pca_ridge_full,     'full',  dict(n_comp=8, alpha=30)),
]

In [ ]:
def inner_loocv_score(X_inner, y_clr_inner, y_cum_inner, builder, kwargs):
    loo = LeaveOneOut()
    preds = np.zeros_like(y_cum_inner)
    for tr, va in loo.split(X_inner):
        pipe = builder(**kwargs)
        pipe.fit(X_inner[tr], y_clr_inner[tr])
        pred_clr = pipe.predict(X_inner[va])
        pred_bins = clr_inverse(pred_clr)
        pred_cum = np.cumsum(pred_bins, axis=1)
        pred_cum = np.clip(pred_cum, 0, 100)
        pred_cum[:, -1] = 100.0
        preds[va] = pred_cum
    return emd_score(y_cum_inner, preds).mean()

outer_loo = LeaveOneOut()
n_samples = len(y_true_cum)
nested_preds = np.zeros_like(y_true_cum)
selected_configs = []

for outer_tr, outer_va in outer_loo.split(np.arange(n_samples)):
    y_clr_inner = y_clr[outer_tr]
    y_cum_inner = y_true_cum[outer_tr]

    # --- inner search: เลือก config ที่ดีที่สุด เห็นแค่ 23 samples (ไม่เห็น outer_va เลย) ---
    best_inner = (np.inf, None)
    for name, builder, source, kwargs in candidates:
        X_inner = sources[source][outer_tr]
        s = inner_loocv_score(X_inner, y_clr_inner, y_cum_inner, builder, kwargs)
        if s < best_inner[0]:
            best_inner = (s, (name, builder, source, kwargs))

    _, (name, builder, source, kwargs) = best_inner
    selected_configs.append(name)

    # --- เทรน config ที่ถูกเลือกด้วย 23 samples เต็ม ๆ แล้วทำนาย sample ที่ถูก hold-out ---
    Xs = sources[source]
    pipe = builder(**kwargs)
    pipe.fit(Xs[outer_tr], y_clr[outer_tr])
    pred_clr = pipe.predict(Xs[outer_va])
    pred_bins = clr_inverse(pred_clr)
    pred_cum = np.cumsum(pred_bins, axis=1)
    pred_cum = np.clip(pred_cum, 0, 100)
    pred_cum[:, -1] = 100.0
    nested_preds[outer_va] = pred_cum

nested_score = emd_score(y_true_cum, nested_preds).mean()
print('Nested CV EMD (ตัวเลขที่เชื่อถือได้จริง):', nested_score)

from collections import Counter
print('ความถี่ config ที่ถูกเลือกในแต่ละ outer fold:', Counter(selected_configs))

In [ ]:
final_pipe = build_pca_ridge_full(n_comp=5, alpha=100)
final_pipe.fit(X_full, y_clr)   # X_full = train_agg.values (feature เต็มชุด)

pred_clr_test = final_pipe.predict(test_agg.values)
pred_bins_test = clr_inverse(pred_clr_test)
pred_cum_test  = np.cumsum(pred_bins_test, axis=1)

pred_cum_test = np.maximum.accumulate(pred_cum_test, axis=1)
pred_cum_test = np.clip(pred_cum_test, 0, 100)
pred_cum_test[:, -1] = 100.0

In [ ]:
submission = pd.DataFrame(pred_cum_test, columns=COLS)
submission.insert(0, 'sample_id', test_df['sample_id'].values)

assert (submission[COLS].diff(axis=1).iloc[:,1:] >= -1e-6).all().all(), 'monotonic check failed'
assert np.allclose(submission[COLS[-1]], 100.0), 'last column must be 100'

submission.to_csv('submission.csv', index=False)
submission.head()

score 123 ไม่work

error analysis

In [ ]:
def build_pca_ridge_original(n_comp=5, alpha=100):
    return Pipeline([
        ('pca', PCA(n_components=n_comp, random_state=0)),
        ('ridge', Ridge(alpha=alpha)),
    ])

loo = LeaveOneOut()
loocv_preds = np.zeros_like(y_true_cum)

for tr_idx, va_idx in loo.split(X_full):
    pipe = build_pca_ridge_original(n_comp=5, alpha=100)
    pipe.fit(X_full[tr_idx], y_clr[tr_idx])
    pred_clr = pipe.predict(X_full[va_idx])
    pred_bins = clr_inverse(pred_clr)
    pred_cum = np.cumsum(pred_bins, axis=1)
    pred_cum = np.clip(pred_cum, 0, 100)
    pred_cum[:, -1] = 100.0
    loocv_preds[va_idx] = pred_cum

per_sample_emd = emd_score(y_true_cum, loocv_preds)

In [ ]:
photo_count = train_img_df.groupby('sample_id').size().reindex(train_df['sample_id']).values
cameras_used = train_img_df.groupby('sample_id')['camera'].apply(lambda x: ','.join(sorted(set(x)))).reindex(train_df['sample_id']).values

error_df = pd.DataFrame({
    'sample_id': train_df['sample_id'].values,
    'emd': per_sample_emd,
    'n_photos': photo_count,
    'cameras': cameras_used,
})
error_df = error_df.sort_values('emd', ascending=False)
print(error_df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

worst_ids = error_df.head(3)['sample_id'].tolist()
best_ids  = error_df.tail(3)['sample_id'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(12,5))
for ax, ids, title in zip(axes, [worst_ids, best_ids], ['Worst 3', 'Best 3']):
    for sid in ids:
        idx = list(train_df['sample_id']).index(sid)
        ax.plot(DIAMS, y_true_cum[idx], 'o-', label=f'{sid} true')
        ax.plot(DIAMS, loocv_preds[idx], 'x--', label=f'{sid} pred')
    ax.set_xscale('log')
    ax.set_title(title)
    ax.set_xlabel('diameter (mm)')
    ax.set_ylabel('cumulative %')
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
d50_true = np.array([np.interp(50, y_true_cum[i], DIAMS) for i in range(len(y_true_cum))])  # ขนาดที่ 50% ผ่าน

print('corr(error, n_photos):', np.corrcoef(per_sample_emd, photo_count)[0,1])
print('corr(error, D50 ของ sample จริง):', np.corrcoef(per_sample_emd, d50_true)[0,1])

error_df['d50'] = d50_true
print(error_df[['sample_id','emd','n_photos','d50']].to_string(index=False))

In [ ]:
deep_dim = 1280
cls_names = (
    [f'glcm_{i}' for i in range(16)] +
    [f'lbp_{i}' for i in range(10)] +
    [f'color_{i}' for i in range(48)] +
    ['edge_density'] +
    ['size_mean','size_std','size_median','size_p25','size_p75','size_max','size_count']
)
base_names = [f'deep_{i}' for i in range(deep_dim)] + cls_names
feature_names = [f'mean_{n}' for n in base_names] + [f'std_{n}' for n in base_names]

final_pipe = build_pca_ridge_original(n_comp=5, alpha=100)
final_pipe.fit(X_full, y_clr)
pca = final_pipe.named_steps['pca']

for i, comp in enumerate(pca.components_):
    top_idx = np.argsort(np.abs(comp))[::-1][:8]
    print(f'PC{i+1}:', [(feature_names[j], round(comp[j],3)) for j in top_idx])

In [ ]:
# ระยะห่างของแต่ละ sample จาก "รูปทรงเฉลี่ย" ของ composition ทั้งหมด (ยิ่งไกลยิ่งเป็น outlier)
target_outlierness = np.linalg.norm(y_clr - y_clr.mean(axis=0), axis=1)

error_df['outlierness'] = error_df['sample_id'].map(
    dict(zip(train_df['sample_id'], target_outlierness))
)
print('corr(emd, outlierness):', np.corrcoef(error_df['emd'], error_df['outlierness'])[0,1])
print(error_df[['sample_id','emd','outlierness']].sort_values('emd', ascending=False).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
for i, sid in enumerate(train_df['sample_id']):
    color = 'red' if sid in ['H038','H516','H037'] else 'lightgray'
    lw = 2.5 if sid in ['H038','H516','H037'] else 1
    ax.plot(DIAMS, y_true_cum[i], color=color, linewidth=lw, label=sid if color=='red' else None)
ax.set_xscale('log')
ax.set_xlabel('diameter (mm)')
ax.set_ylabel('cumulative % (true)')
ax.legend()
plt.show()